# Smart Market Intelligence Platform v2
### Interactive ML Pipeline Notebook

This notebook walks through the **complete training pipeline** step by step — run each cell individually to inspect intermediate results, dataframes, and charts inline.

| Step | What happens |
|---|---|
| 1 | Install & import dependencies |
| 2 | Load & validate raw data |
| 3 | Clean & preprocess |
| 4 | RFM feature engineering |
| 5 | K-Means customer segmentation |
| 6 | Daily sales aggregation + lag features |
| 7 | Sales forecasting with TimeSeriesSplit |
| 8 | Model evaluation & charts |
| 9 | Model versioning & metadata export |
| 10 | API endpoint testing |

> **Note:** `api/`, `db/`, `dashboard.py` stay as `.py` files — they are imported as modules or run by frameworks (uvicorn / streamlit). This notebook covers everything that was previously in `run_pipeline.py`.

## Step 1 — Install & Import Dependencies

In [1]:
import os, json, warnings
warnings.filterwarnings("ignore")

# Load .env (DATABASE_URL etc.)
from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")   # headless — swap to "inline" if using classic Jupyter
%matplotlib inline
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, silhouette_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
import joblib

print("All imports OK")
print(f"  numpy      {np.__version__}")
print(f"  pandas     {pd.__version__}")

All imports OK
  numpy      2.4.2
  pandas     3.0.1


## Step 2 — Load & Validate Raw Data

The raw CSV has 541,909 rows. We run four fail-fast guardrails **before** touching any data:
1. Not empty
2. Required columns present
3. Negative `UnitPrice` < 1% of rows (data entry errors, not systemic corruption)
4. No `Quantity` > 100,000 (physiologically impossible single transaction)

In [2]:
RAW_PATH = "data/Online Retail.csv"

df_raw = pd.read_csv(RAW_PATH, encoding="ISO-8859-1")
print(f"Loaded {len(df_raw):,} rows, {df_raw.shape[1]} columns")

# --- Guardrails ---
REQUIRED_COLUMNS = ["InvoiceNo","StockCode","Description","Quantity",
                    "InvoiceDate","UnitPrice","CustomerID","Country"]
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"
print(f"[OK] Schema valid — {df_raw.shape[1]} columns present")

assert not df_raw.empty, "Dataset is empty!"
print(f"[OK] Not empty — {len(df_raw):,} rows")

neg_price = df_raw[df_raw["UnitPrice"] < 0]
neg_pct = len(neg_price) / len(df_raw)
assert neg_pct < 0.01, f"Too many negative UnitPrice rows: {neg_pct:.2%}"
print(f"[OK] Negative UnitPrice: {len(neg_price)} rows ({neg_pct:.4%}) — within tolerance")

extreme_qty = df_raw[df_raw["Quantity"] > 100_000]
assert len(extreme_qty) == 0, f"Found {len(extreme_qty)} rows with Quantity > 100,000"
print(f"[OK] Quantity range valid")

# Preview
df_raw.head(3)

Loaded 541,909 rows, 8 columns
[OK] Schema valid — 8 columns present
[OK] Not empty — 541,909 rows
[OK] Negative UnitPrice: 2 rows (0.0004%) — within tolerance
[OK] Quantity range valid


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom


## Step 3 — Data Cleaning & Preprocessing

In [3]:
df = df_raw.copy()

before = len(df)
df = df.dropna(subset=["CustomerID"])
print(f"Dropped {before - len(df):,} rows (missing CustomerID) → {len(df):,} remain")

before = len(df)
df = df[df["Quantity"] > 0]
print(f"Dropped {before - len(df):,} rows (Quantity <= 0 / returns) → {len(df):,} remain")

before = len(df)
df = df[df["UnitPrice"] > 0]
print(f"Dropped {before - len(df):,} rows (UnitPrice <= 0) → {len(df):,} remain")

df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = df["CustomerID"].astype(int)

print(f"\nFinal cleaned dataset: {len(df):,} rows")
df.to_csv("data/cleaned_data.csv", index=False)
print("Saved → data/cleaned_data.csv")

df[["InvoiceNo","CustomerID","InvoiceDate","Quantity","UnitPrice","TotalAmount","Country"]].head(5)

Dropped 135,080 rows (missing CustomerID) → 406,829 remain
Dropped 8,905 rows (Quantity <= 0 / returns) → 397,924 remain
Dropped 40 rows (UnitPrice <= 0) → 397,884 remain

Final cleaned dataset: 397,884 rows
Saved → data/cleaned_data.csv


,InvoiceNo,CustomerID,InvoiceDate,Quantity,UnitPrice,TotalAmount,Country
0,536365,17850,2010-12-01 08:26:00,6,2.55,15.30,United Kingdom
1,536365,17850,2010-12-01 08:26:00,6,3.39,20.34,United Kingdom
2,536365,17850,2010-12-01 08:26:00,8,2.75,22.00,United Kingdom
3,536365,17850,2010-12-01 08:26:00,6,3.39,20.34,United Kingdom
4,536365,17850,2010-12-01 08:26:00,6,3.39,20.34,United Kingdom


## Step 4 — RFM Feature Engineering

**Recency**: Days since last purchase (lower = more recent = better)  
**Frequency**: Number of unique invoices  
**Monetary**: Total spend across all invoices  

We apply `log1p` before scaling to compress the long tail of high-spending customers.

In [4]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency   = ("InvoiceDate",  lambda x: (snapshot_date - x.max()).days),
    Frequency = ("InvoiceNo",    "nunique"),
    Monetary  = ("TotalAmount",  "sum"),
).reset_index()

print(f"RFM table: {len(rfm):,} customers")
print(rfm[["Recency","Frequency","Monetary"]].describe().round(2))

# Log transform to reduce skew, then scale
rfm_log = np.log1p(rfm[["Recency","Frequency","Monetary"]])
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

rfm.head()

RFM table: 4,338 customers
       Recency  Frequency   Monetary
count  4338.00    4338.00    4338.00
mean     92.54       4.27    2054.27
std     100.01       7.70    8989.23
min       1.00       1.00       3.75
25%      18.00       1.00     307.41
50%      51.00       2.00     674.48
75%     142.00       5.00    1661.74
max     374.00     209.00  280206.02


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


## Step 5 — Customer Segmentation with K-Means

We train K-Means with **k=3** and evaluate with **Silhouette Score** (–1 to +1, higher = tighter clusters).  
Segments are labelled by comparing each cluster's mean RFM values.

In [5]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

sil = silhouette_score(rfm_scaled, rfm["Cluster"])
print(f"Silhouette Score: {sil:.4f}  (above 0.3 = acceptable separation)")

# Label segments by cluster means
cluster_means = rfm.groupby("Cluster")[["Recency","Frequency","Monetary"]].mean()
print("\nCluster means:")
print(cluster_means.round(1))

# Assign human-readable labels:
#   Loyal  = lowest recency (most recent), high frequency
#   Whale  = highest monetary
#   At-Risk = highest recency (longest since purchase)
labels = {}
labels[cluster_means["Recency"].idxmin()]   = "Loyal"
labels[cluster_means["Monetary"].idxmax()]  = "Whale"
for k in range(3):
    if k not in labels:
        labels[k] = "At-Risk"
rfm["Segment"] = rfm["Cluster"].map(labels)
print("\nSegment counts:")
print(rfm["Segment"].value_counts())

# Save artefacts
os.makedirs("models", exist_ok=True)
joblib.dump(kmeans,  "models/kmeans_model.pkl")
joblib.dump(scaler,  "models/scaler.pkl")
print("\nSaved → models/kmeans_model.pkl, models/scaler.pkl")

# Save customer segments CSV
rfm.to_csv("data/customer_segments.csv", index=False)

# Pie chart
fig = px.pie(rfm, names="Segment", title="Customer Segments",
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

Silhouette Score: 0.3365  (above 0.3 = acceptable separation)

Cluster means:
         Recency  Frequency  Monetary
Cluster                              
0           17.1       13.3    7905.4
1          167.4        1.4     362.5
2           44.2        3.4    1265.1

Segment counts:
Segment
At-Risk    3568
Whale       770
Name: count, dtype: int64

Saved → models/kmeans_model.pkl, models/scaler.pkl


## Step 6 — Daily Sales Aggregation & Lag Features

Aggregate to one row per day, then add:
- **Calendar features**: Year, Month, Day, DayOfWeek, Quarter, DayOfYear, WeekOfYear, IsWeekend
- **Context features**: NumTransactions, NumCustomers, TotalQuantity  
- **Lag features**: Sales 1, 7, 14 days ago (temporal dependency)
- **Rolling features**: 7-day and 30-day moving averages (trend signal)

In [6]:
df["Date"] = df["InvoiceDate"].dt.date

daily = df.groupby("Date").agg(
    Sales            = ("TotalAmount",  "sum"),
    NumTransactions  = ("InvoiceNo",    "nunique"),
    NumCustomers     = ("CustomerID",   "nunique"),
    TotalQuantity    = ("Quantity",     "sum"),
).reset_index()
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values("Date").reset_index(drop=True)

# Calendar features
daily["Year"]      = daily["Date"].dt.year
daily["Month"]     = daily["Date"].dt.month
daily["Day"]       = daily["Date"].dt.day
daily["DayOfWeek"] = daily["Date"].dt.dayofweek
daily["Quarter"]   = daily["Date"].dt.quarter
daily["DayOfYear"] = daily["Date"].dt.dayofyear
daily["WeekOfYear"]= daily["Date"].dt.isocalendar().week.astype(int)
daily["IsWeekend"] = (daily["DayOfWeek"] >= 5).astype(int)

# Lag features
daily["Sales_Lag1"]  = daily["Sales"].shift(1)
daily["Sales_Lag7"]  = daily["Sales"].shift(7)
daily["Sales_Lag14"] = daily["Sales"].shift(14)

# Rolling means
daily["Sales_MA7"]   = daily["Sales"].shift(1).rolling(7).mean()
daily["Sales_MA30"]  = daily["Sales"].shift(1).rolling(30).mean()

daily = daily.dropna().reset_index(drop=True)
daily.to_csv("data/daily_sales.csv", index=False)

print(f"Daily sales table: {len(daily)} rows x {daily.shape[1]} columns")
print(f"Date range: {daily['Date'].min().date()} to {daily['Date'].max().date()}")
print(f"Sales range: £{daily['Sales'].min():,.0f} to £{daily['Sales'].max():,.0f}")

# Sales trend chart
fig = px.line(daily, x="Date", y="Sales", title="Daily Sales Trend",
              labels={"Sales": "Sales (GBP)"},
              color_discrete_sequence=["#2196F3"])
fig.show()

daily.tail(5)

Daily sales table: 275 rows x 18 columns
Date range: 2011-01-16 to 2011-12-09
Sales range: £3,457 to £184,349


,Date,Sales,NumTransactions,NumCustomers,TotalQuantity,Year,Month,Day,DayOfWeek,Quarter,DayOfYear,WeekOfYear,IsWeekend,Sales_Lag1,Sales_Lag7,Sales_Lag14,Sales_MA7,Sales_MA30
270,2011-12-05,58202.21,116,105,38428,2011,12,5,0,4,339,49,0,20375.96,17300.96,36751.25,38441.311429,43482.986333
271,2011-12-06,46144.04,110,103,27827,2011,12,6,1,4,340,49,0,58202.21,51831.67,30190.92,44284.347143,44321.441000
272,2011-12-07,69354.21,104,94,41102,2011,12,7,2,4,341,49,0,46144.04,48851.68,45333.13,43471.828571,44888.482000
273,2011-12-08,50519.41,113,105,27551,2011,12,8,3,4,342,49,0,69354.21,41481.23,49664.89,46400.761429,45909.132333
274,2011-12-09,184349.28,41,35,90582,2011,12,9,4,4,343,49,0,50519.41,44533.99,71979.93,47691.930000,45570.709000


## Step 7 — Sales Forecasting with Random Forest (TimeSeriesSplit)

**Why TimeSeriesSplit instead of random KFold?**  
Random splitting lets future data leak into training — the model learns patterns it shouldn't know at prediction time. `TimeSeriesSplit` always trains on the past and tests on the future, matching real deployment conditions.

We benchmark three models and pick the best by **MAPE** (scale-independent, board-friendly metric).

In [7]:
FEATURE_COLS = [
    "Year","Month","Day","DayOfWeek","Quarter","DayOfYear","WeekOfYear","IsWeekend",
    "NumTransactions","NumCustomers","TotalQuantity",
    "Sales_Lag1","Sales_Lag7","Sales_Lag14","Sales_MA7","Sales_MA30",
]
available = [c for c in FEATURE_COLS if c in daily.columns]
X = daily[available].values
y = daily["Sales"].values

print(f"Feature matrix: {X.shape[0]} rows x {X.shape[1]} features")

tscv = TimeSeriesSplit(n_splits=5)

models_to_bench = {
    "Linear":       LinearRegression(),
    "Ridge":        Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=10,
                                          min_samples_split=5, random_state=42),
}

bench_results = {}
for name, mdl in models_to_bench.items():
    rmse_list, mape_list, r2_list = [], [], []
    for tr_idx, te_idx in tscv.split(X):
        mdl.fit(X[tr_idx], y[tr_idx])
        pred = mdl.predict(X[te_idx])
        rmse_list.append(np.sqrt(mean_squared_error(y[te_idx], pred)))
        mask = y[te_idx] != 0
        mape_list.append(np.mean(np.abs((y[te_idx][mask]-pred[mask])/y[te_idx][mask]))*100)
        ss_res = np.sum((y[te_idx]-pred)**2)
        ss_tot = np.sum((y[te_idx]-np.mean(y[te_idx]))**2)
        r2_list.append(1 - ss_res/ss_tot if ss_tot else 0)
    bench_results[name] = {
        "RMSE": round(np.mean(rmse_list), 2),
        "MAPE": round(np.mean(mape_list), 2),
        "R2":   round(np.mean(r2_list),   4),
    }

# Display as a nice table
bench_df = pd.DataFrame(bench_results).T.sort_values("MAPE")
bench_df["Best"] = ["<-- BEST" if i == 0 else "" for i in range(len(bench_df))]
print("\nBenchmark results (sorted by MAPE):")
bench_df

Feature matrix: 275 rows x 16 features

Benchmark results (sorted by MAPE):


,RMSE,MAPE,R2,Best
Linear,6676.71,13.11,0.7960,<-- BEST
RandomForest,8411.78,13.40,0.7014,
Ridge,6882.09,14.43,0.7680,


## Step 8 — Model Evaluation & Charts

Train the best model (selected by MAPE) on the full dataset, evaluate on the final holdout fold, and display:
- Predicted vs Actual sales chart
- Residuals distribution
- Feature importance (top 10)

In [8]:
# --- Select best model by MAPE ---
best_name = min(bench_results, key=lambda k: bench_results[k]["MAPE"])
print(f"Best model: {best_name}  (MAPE={bench_results[best_name]['MAPE']:.2f}%)")

# Map name back to a fresh model instance (with tuned params for RF)
model_map = {
    "Linear":       LinearRegression(),
    "Ridge":        Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=10,
                                          min_samples_split=5, random_state=42),
}
best_model = model_map[best_name]
best_model.fit(X, y)

# Holdout evaluation on last fold
splits = list(tscv.split(X))
_, te_idx = splits[-1]
y_test = y[te_idx]
y_pred = best_model.predict(X[te_idx])

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
mask = y_test != 0
mape = np.mean(np.abs((y_test[mask]-y_pred[mask])/y_test[mask]))*100
ss_res = np.sum((y_test-y_pred)**2); ss_tot = np.sum((y_test-np.mean(y_test))**2)
r2 = 1 - ss_res/ss_tot

print(f"\nHoldout metrics (last fold):")
print(f"  RMSE : GBP {rmse:,.2f}")
print(f"  MAE  : GBP {mae:,.2f}")
print(f"  MAPE : {mape:.2f}%")
print(f"  R2   : {r2:.4f}")

# Predicted vs Actual
dates_test = daily["Date"].iloc[te_idx].values
fig = go.Figure()
fig.add_trace(go.Scatter(x=dates_test, y=y_test, name="Actual",  line=dict(color="#2196F3")))
fig.add_trace(go.Scatter(x=dates_test, y=y_pred, name="Predicted", line=dict(color="#FF5722", dash="dash")))
fig.update_layout(title=f"Predicted vs Actual Sales ({best_name})",
                  xaxis_title="Date", yaxis_title="Sales (GBP)")
fig.show()

# Residuals histogram
residuals = y_test - y_pred
fig2 = px.histogram(x=residuals, nbins=30, title="Residual Distribution",
                    labels={"x":"Residual (GBP)"},
                    color_discrete_sequence=["#FF5722"])
fig2.add_vline(x=0, line_dash="dash", line_color="black")
fig2.show()

# Feature importance (Ridge uses coef magnitudes)
if hasattr(best_model, "feature_importances_"):
    imp = best_model.feature_importances_
else:
    imp = np.abs(best_model.coef_)
pairs = sorted(zip(available, imp), key=lambda x: x[1], reverse=True)[:10]
fig3 = px.bar(x=[p[1] for p in pairs], y=[p[0] for p in pairs],
              orientation="h", title="Top 10 Feature Importances",
              labels={"x":"Importance","y":"Feature"},
              color_discrete_sequence=["#2196F3"])
fig3.update_yaxes(autorange="reversed")
fig3.show()

Best model: Linear  (MAPE=13.11%)

Holdout metrics (last fold):
  RMSE : GBP 6,715.77
  MAE  : GBP 4,309.51
  MAPE : 9.49%
  R2   : 0.9267


## Step 9 — Model Versioning & Metadata Export

Save two copies of the model:
- `best_model.pkl` — what the API loads at startup
- `sales_model_v1.pkl` — timestamped backup for rollback

`metadata.json` records everything needed to reproduce or audit this run.

In [9]:
from datetime import datetime

os.makedirs("models", exist_ok=True)

# Primary model (API loads this)
joblib.dump(best_model, "models/best_model.pkl")
print("Saved → models/best_model.pkl")

# Versioned backup
joblib.dump(best_model, "models/sales_model_v1.pkl")
print("Saved → models/sales_model_v1.pkl")

# Metadata
metadata = {
    "version":        "v2",
    "model_name":     best_name,
    "training_date":  datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "feature_names":  available,
    "n_features":     len(available),
    "n_training_samples": len(X),
    "metrics": {
        "RMSE": round(rmse, 2),
        "MAE":  round(mae,  2),
        "MAPE": round(mape, 2),
        "R2":   round(r2,   4),
    },
    "hyperparameters": (
        best_model.get_params() if hasattr(best_model, "get_params") else {}
    ),
    "residual_analysis": {
        "mean_residual":          round(float(np.mean(residuals)), 2),
        "std_residual":           round(float(np.std(residuals)),  2),
        "pct_over_20pct_error":   round(float(np.nanmean(
            np.abs(residuals / np.where(y_test!=0, y_test, np.nan)) > 0.2
        ) * 100), 2),
    },
    "benchmark_comparison": bench_results,
}

with open("models/metadata.json", "w") as fh:
    json.dump(metadata, fh, indent=2)
print("Saved → models/metadata.json")

print("\n--- Metadata summary ---")
print(f"  Model   : {metadata['model_name']}")
print(f"  RMSE    : GBP {metadata['metrics']['RMSE']:,.2f}")
print(f"  MAPE    : {metadata['metrics']['MAPE']:.2f}%")
print(f"  R2      : {metadata['metrics']['R2']:.4f}")
print(f"  Trained : {metadata['training_date']}")

Saved → models/best_model.pkl
Saved → models/sales_model_v1.pkl
Saved → models/metadata.json

--- Metadata summary ---
  Model   : Linear
  RMSE    : GBP 6,715.77
  MAPE    : 9.49%
  R2      : 0.9267
  Trained : 2026-02-24 18:03:36


## Step 10 — API Endpoint Testing

Test the live FastAPI server at `http://127.0.0.1:8000`.  
Make sure the server is running first:
```bash
.\venv\Scripts\python.exe -m uvicorn api.app:app --reload
```

In [12]:
import requests

BASE = "http://127.0.0.1:8000"

# ── Health check ──────────────────────────────────────────────
r = requests.get(f"{BASE}/health")
print("GET /health")
print(f"  Status : {r.status_code}")
health = r.json()
for k, v in health.items():
    print(f"  {k:<18}: {v}")

# ── Benchmark comparison ──────────────────────────────────────
print("\nGET /benchmark")
r = requests.get(f"{BASE}/benchmark")
bench = r.json()
for model, metrics in bench.items():
    print(f"  {model:<15} RMSE={metrics['RMSE']:>8,.2f}  "
          f"MAPE={metrics['MAPE']:>6.2f}%  R2={metrics['R2']:.4f}")

# ── 7-day forecast ────────────────────────────────────────────
print("\nPOST /predict  (days=7)")
r = requests.post(f"{BASE}/predict", json={"days": 7})
pred = r.json()
print(f"  Model version  : {pred['model_version']}")
print(f"  Avg prediction : GBP {pred['avg_prediction']:,.2f}")
print(f"  Latency        : {pred['request_latency_ms']:.1f} ms")

# Plot forecast
import plotly.graph_objects as go
from datetime import datetime, timedelta
dates = [datetime.today() + timedelta(days=i) for i in range(7)]
fig = go.Figure(go.Bar(
    x=[d.strftime("%a %d %b") for d in dates],
    y=pred["forecast_values"],
    marker_color="#2196F3",
))
fig.update_layout(title="7-Day Sales Forecast from API",
                  xaxis_title="Day", yaxis_title="Predicted Sales (GBP)")
fig.show()

# ── Drift monitor ─────────────────────────────────────────────
print("\nGET /drift")
r = requests.get(f"{BASE}/drift")
drift = r.json()
print(f"  Training mean sales : GBP {drift['training_mean']:,.2f}")
print(f"  Rolling mean (50)   : GBP {drift['rolling_mean_50']:,.2f}")
print(f"  Drift               : {drift['drift_pct']:+.2f}%")
print(f"  Drift flag          : {drift['drift_flag']}")

GET /health
  Status : 200
  status            : ok
  model_version     : v2-Linear-2026-02-24
  db_connected      : True
  api_version       : 2.0.0

GET /benchmark
  Linear          RMSE=6,274.73  MAPE= 14.08%  R2=0.7912
  Ridge           RMSE=6,292.69  MAPE= 13.08%  R2=0.7878
  RandomForest    RMSE=8,528.42  MAPE= 14.74%  R2=0.5989

POST /predict  (days=7)
  Model version  : v2-Linear-2026-02-24
  Avg prediction : GBP 32,377.13
  Latency        : 7.3 ms



GET /drift
  Training mean sales : GBP 29,368.57
  Rolling mean (50)   : GBP 29,368.57
  Drift               : +0.00%
  Drift flag          : False
